# Test du Service de Visualisation

Ce notebook teste toutes les fonctionnalités du service de visualisation avec des données factices de logs de sécurité.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import random
import sys
import os

# Ajouter le chemin vers les services
sys.path.append(os.path.join(os.getcwd(), 'src', 'app', 'services'))

from visualization_service import VisualizationService

In [2]:
# Initialisation du service
viz_service = VisualizationService()
print("Service de visualisation initialisé avec succès!")

Service de visualisation initialisé avec succès!


In [3]:
# Génération de données factices de logs de sécurité
def generate_fake_security_logs(n_records=1000):
    """
    Génère des données factices simulant des logs de pare-feu/sécurité
    """
    protocols = ['TCP', 'UDP', 'ICMP', 'HTTP', 'HTTPS', 'FTP', 'SSH']
    actions = ['ALLOW', 'DENY', 'DROP', 'ACCEPT', 'REJECT']
    sources = [f"192.168.{random.randint(1,255)}.{random.randint(1,255)}" for _ in range(50)]
    destinations = [f"10.0.{random.randint(1,255)}.{random.randint(1,255)}" for _ in range(30)]
    
    # Types d'événements de sécurité
    event_types = [
        'Connection', 'Intrusion Attempt', 'Port Scan', 'DDoS',
        'Authentication Failure', 'Malware Detection', 'Data Exfiltration'
    ]
    
    data = []
    
    for i in range(n_records):
        # Timestamp avec distribution réaliste (plus d'activité en journée)
        base_date = datetime.now() - timedelta(days=random.randint(0, 30))
        hour_weight = [2, 1, 1, 1, 2, 4, 6, 8, 10, 12, 14, 12, 10, 8, 6, 4, 3, 2, 2, 1, 1, 1, 1, 1]
        hour = np.random.choice(range(24), p=np.array(hour_weight)/sum(hour_weight))
        timestamp = base_date.replace(hour=hour, minute=random.randint(0,59), second=random.randint(0,59))
        
        # Simulation d'anomalies (10% des données)
        is_anomaly = random.random() < 0.1
        severity = 'HIGH' if is_anomaly else random.choices(
            ['LOW', 'MEDIUM', 'HIGH'], weights=[50, 35, 15]
        )[0]
        
        record = {
            'timestamp': timestamp,
            'source_ip': random.choice(sources),
            'destination_ip': random.choice(destinations),
            'source_port': random.randint(1024, 65535) if random.random() > 0.3 else random.choice([22, 80, 443, 21, 25]),
            'destination_port': random.choice([22, 80, 443, 21, 25, 53, 3389, 3306, 5432]),
            'protocol_deduced': random.choice(protocols),
            'action': random.choice(actions),
            'bytes_sent': random.randint(64, 100000),
            'bytes_received': random.randint(64, 100000),
            'duration': random.uniform(0.1, 300.0),
            'event_type': random.choice(event_types),
            'severity': severity,
            'is_anomaly': is_anomaly,
            'threat_score': random.uniform(0, 10) if is_anomaly else random.uniform(0, 3),
            'country': random.choice(['US', 'FR', 'CN', 'RU', 'DE', 'UK', 'CA']),
            'user_agent': random.choice([
                'Mozilla/5.0', 'Chrome/91.0', 'Safari/14.0', 'Bot/1.0', None
            ])
        }
        data.append(record)
    
    return pd.DataFrame(data)

# Génération des données
df = generate_fake_security_logs(2000)
print(f"Générées {len(df)} entrées de logs factices")
print(f"Colonnes disponibles: {list(df.columns)}")
print(f"Période couverte: {df['timestamp'].min()} à {df['timestamp'].max()}")

Générées 2000 entrées de logs factices
Colonnes disponibles: ['timestamp', 'source_ip', 'destination_ip', 'source_port', 'destination_port', 'protocol_deduced', 'action', 'bytes_sent', 'bytes_received', 'duration', 'event_type', 'severity', 'is_anomaly', 'threat_score', 'country', 'user_agent']
Période couverte: 2026-01-31 00:47:41.808705 à 2026-03-02 22:34:21.784911


In [11]:
# Fonctions dédiées: une fonction par graphique + sauvegarde HTML

def save_html_output(content: str, file_name: str, label: str) -> None:
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"✅ {label} HTML généré")
    print(f"📁 Sauvegardé dans '{file_name}'")


def build_protocol_chart(df: pd.DataFrame) -> str:
    return viz_service.create_protocol_distribution_chart(df, to_html=True)


def build_temporal_chart(df: pd.DataFrame) -> str:
    return viz_service.create_temporal_activity_chart(df, to_html=True)


def build_security_dashboard(df: pd.DataFrame, security_stats: dict) -> str:
    return viz_service.create_security_dashboard(df, security_stats, to_html=True)


def build_anomaly_chart(df: pd.DataFrame, anomaly_data: dict) -> str:
    return viz_service.create_anomaly_detection_chart(df, anomaly_data, threshold=5.0, to_html=True)


def build_network_topology(df: pd.DataFrame, max_nodes: int = 30) -> str:
    return viz_service.create_network_topology_chart(df, max_nodes=max_nodes, to_html=True)


def build_colors_test_chart(df: pd.DataFrame) -> str:
    import plotly.express as px

    action_counts = df['action'].value_counts()
    fig_colors = px.bar(
        x=action_counts.index,
        y=action_counts.values,
        title="Test des Couleurs - Distribution des Actions",
        color=action_counts.index,
        color_discrete_map=viz_service.color_schemes['actions'],
    )
    return fig_colors.to_html()

In [4]:
# Aperçu des données
print("=== APERÇU DES DONNÉES ===")
df.head()

=== APERÇU DES DONNÉES ===


,timestamp,source_ip,destination_ip,source_port,destination_port,protocol_deduced,action,bytes_sent,bytes_received,duration,event_type,severity,is_anomaly,threat_score,country,user_agent
0,2026-02-08 08:31:54.719004,192.168.178.6,10.0.133.137,15628,3306,ICMP,REJECT,64588,50740,30.190343,DDoS,LOW,False,2.888985,CA,Safari/14.0
1,2026-02-27 19:46:48.719463,192.168.130.234,10.0.80.197,48248,80,HTTP,ALLOW,88854,81927,225.455609,Data Exfiltration,LOW,False,2.542630,US,Chrome/91.0
2,2026-02-24 12:11:50.719618,192.168.239.132,10.0.244.190,6837,80,FTP,DROP,97316,8012,225.829271,Intrusion Attempt,LOW,False,1.348168,DE,Mozilla/5.0
3,2026-02-01 14:46:06.719761,192.168.142.106,10.0.91.185,443,25,HTTP,DROP,30610,73862,92.020310,Data Exfiltration,MEDIUM,False,2.054463,UK,Chrome/91.0
4,2026-02-09 11:47:35.720042,192.168.103.94,10.0.9.88,80,80,FTP,DENY,2955,66912,240.691512,Authentication Failure,LOW,False,2.366292,US,Mozilla/5.0


In [5]:
# Test 1: Distribution des protocoles
print("=== TEST 1: DISTRIBUTION DES PROTOCOLES ===")

html_protocol = build_protocol_chart(df)
save_html_output(html_protocol, "protocol_chart.html", "Graphique protocoles")

print(f"Protocoles détectés: {df['protocol_deduced'].value_counts().to_dict()}")

=== TEST 1: DISTRIBUTION DES PROTOCOLES ===


✅ Graphique HTML généré avec succès!
📁 Graphique sauvegardé dans 'protocol_chart.html'
Protocoles détectés: {'HTTP': 298, 'ICMP': 293, 'UDP': 293, 'HTTPS': 290, 'TCP': 284, 'FTP': 271, 'SSH': 271}


In [ ]:
# Test 2: Activité temporelle
print("=== TEST 2: ACTIVITÉ TEMPORELLE ===")

html_temporal = build_temporal_chart(df)
save_html_output(html_temporal, "temporal_chart.html", "Graphique activité temporelle")

print(f"Heures les plus actives: {df['timestamp'].dt.hour.value_counts().head(3).to_dict()}")

=== TEST 2: ACTIVITÉ TEMPORELLE ===
✅ Graphique temporal HTML généré!
📁 Graphique sauvegardé dans 'temporal_chart.html'
Heures les plus actives: {10: 240, 11: 222, 9: 220}


In [ ]:
# Test 3: Dashboard de sécurité
print("=== TEST 3: DASHBOARD DE SÉCURITÉ ===")

# Préparation des statistiques pour le dashboard
security_stats = {
    'total_events': len(df),
    'unique_sources': df['source_ip'].nunique(),
    'unique_destinations': df['destination_ip'].nunique(),
    'blocked_connections': len(df[df['action'].isin(['DENY', 'DROP', 'REJECT'])]),
    'allowed_connections': len(df[df['action'].isin(['ALLOW', 'ACCEPT'])]),
    'high_severity_events': len(df[df['severity'] == 'HIGH']),
    'anomalies_detected': len(df[df['is_anomaly'] == True]),
    'avg_threat_score': df['threat_score'].mean(),
    'top_source_countries': df['country'].value_counts().head(5).to_dict(),
    'most_targeted_ports': df['destination_port'].value_counts().head(5).to_dict()
}

html_dashboard = build_security_dashboard(df, security_stats)
save_html_output(html_dashboard, "dashboard_chart.html", "Dashboard de sécurité")

print("Statistiques principales:")
for key, value in security_stats.items():
    if isinstance(value, dict):
        print(f"  {key}: {dict(list(value.items())[:3])}...")
    elif isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

=== TEST 3: DASHBOARD DE SÉCURITÉ ===
✅ Dashboard de sécurité HTML généré!
📁 Dashboard sauvegardé dans 'dashboard_chart.html'
Statistiques principales:
  total_events: 2000
  unique_sources: 50
  unique_destinations: 30
  blocked_connections: 1195
  allowed_connections: 805
  high_severity_events: 479
  anomalies_detected: 195
  avg_threat_score: 1.88
  top_source_countries: {'RU': 295, 'FR': 294, 'US': 294}...
  most_targeted_ports: {25: 236, 3389: 234, 80: 230}...


In [ ]:
# Test 4: Détection d'anomalies
print("=== TEST 4: DÉTECTION D'ANOMALIES ===")

anomaly_data = {
    'anomaly_scores': df['threat_score'].values,
    'timestamps': df['timestamp'].values,
    'threshold': 5.0,
    'predictions': (df['threat_score'] > 5.0).astype(int).values,
}

html_anomaly = build_anomaly_chart(df, anomaly_data)
save_html_output(html_anomaly, "anomaly_chart.html", "Graphique détection d'anomalies")

print(f"Anomalies détectées: {sum(anomaly_data['predictions'])} sur {len(df)} événements")
print(f"Seuil utilisé: {anomaly_data['threshold']}")
print(f"Score moyen: {np.mean(anomaly_data['anomaly_scores']):.2f}")

=== TEST 4: DÉTECTION D'ANOMALIES ===


Erreur création graphique anomalies: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


✅ Graphique de détection d'anomalies HTML généré!
📁 Graphique sauvegardé dans 'anomaly_chart.html'
Anomalies détectées: 107 sur 2000 événements
Seuil utilisé: 5.0
Score moyen: 1.88


In [ ]:
# Test 5: Topologie réseau
print("=== TEST 5: TOPOLOGIE RÉSEAU ===")

network_data = df[['source_ip', 'destination_ip', 'protocol_deduced', 'bytes_sent']].copy()
connections = network_data.groupby(['source_ip', 'destination_ip', 'protocol_deduced']).agg({
    'bytes_sent': 'sum'
}).reset_index()
connections.columns = ['source', 'target', 'protocol', 'weight']
top_connections = connections.nlargest(50, 'weight')

html_network = build_network_topology(df, max_nodes=30)
save_html_output(html_network, "network_chart.html", "Topologie réseau")

print(f"Connexions principales analysées: {len(top_connections)}")
print(f"IPs sources uniques: {df['source_ip'].nunique()}")
print(f"IPs destinations uniques: {df['destination_ip'].nunique()}")

=== TEST 5: TOPOLOGIE RÉSEAU ===
✅ Topologie réseau HTML générée!
📁 Topologie sauvegardée dans 'network_chart.html'
Connexions principales analysées: 50
IPs sources uniques: 50
IPs destinations uniques: 30


In [10]:
# Test 6: Métriques et cartes (simulation Streamlit)
print("=== TEST 6: MÉTRIQUES DE SÉCURITÉ ===")

# Simulation des métriques qui seraient affichées dans Streamlit
metrics = {
    'Total Events': len(df),
    'Blocked': len(df[df['action'].isin(['DENY', 'DROP'])]),
    'Allowed': len(df[df['action'].isin(['ALLOW', 'ACCEPT'])]),
    'High Severity': len(df[df['severity'] == 'HIGH']),
    'Anomalies': len(df[df['is_anomaly'] == True]),
    'Avg Threat Score': f"{df['threat_score'].mean():.2f}",
    'Unique Sources': df['source_ip'].nunique(),
    'Unique Destinations': df['destination_ip'].nunique(),
    'Most Active Hour': df['timestamp'].dt.hour.mode()[0],
    'Top Protocol': df['protocol_deduced'].mode()[0],
    'Data Volume': f"{(df['bytes_sent'].sum() + df['bytes_received'].sum()) / 1024 / 1024:.1f} MB"
}

print("Métriques de sécurité calculées:")
for metric, value in metrics.items():
    print(f"  📊 {metric}: {value}")

# Test des pourcentages de blocage par protocole
protocol_stats = df.groupby('protocol_deduced')['action'].apply(
    lambda x: {
        'total': len(x),
        'blocked': len(x[x.isin(['DENY', 'DROP', 'REJECT'])]),
        'allowed': len(x[x.isin(['ALLOW', 'ACCEPT'])]),
        'block_rate': len(x[x.isin(['DENY', 'DROP', 'REJECT'])]) / len(x) * 100
    }
).to_dict()

print("\nTaux de blocage par protocole:")
for protocol, stats in protocol_stats.items():
    print(f"  🔒 {protocol}: {stats['block_rate']:.1f}% bloqués ({stats['blocked']}/{stats['total']})")

=== TEST 6: MÉTRIQUES DE SÉCURITÉ ===
Métriques de sécurité calculées:
  📊 Total Events: 2000
  📊 Blocked: 784
  📊 Allowed: 805
  📊 High Severity: 479
  📊 Anomalies: 195
  📊 Avg Threat Score: 1.88
  📊 Unique Sources: 50
  📊 Unique Destinations: 30
  📊 Most Active Hour: 10
  📊 Top Protocol: HTTP
  📊 Data Volume: 188.9 MB

Taux de blocage par protocole:


TypeError: 'float' object is not subscriptable

In [ ]:
# Test 7: Test des palettes de couleurs et thèmes
print("=== TEST 7: PALETTES DE COULEURS ===")

print("Palettes disponibles dans le service:")
for theme, colors in viz_service.color_schemes.items():
    if isinstance(colors, dict):
        print(f"  🎨 {theme.upper()}: {list(colors.keys())}")
    else:
        print(f"  🎨 {theme.upper()}: {len(colors)} couleurs")

html_colors = build_colors_test_chart(df)
save_html_output(html_colors, "colors_test_chart.html", "Test des couleurs")
print("✅ Test des couleurs réussi!")

In [ ]:
# Résumé final des tests
print("\n" + "="*50)
print("🎉 RÉSUMÉ DES TESTS DU SERVICE DE VISUALISATION")
print("="*50)

tests_results = [
    ("✅ Distribution des protocoles", "Graphiques en secteurs et barres"),
    ("✅ Activité temporelle", "Analyse par heure, jour, heatmap"),
    ("✅ Dashboard de sécurité", "Vue d'ensemble complète"),
    ("✅ Détection d'anomalies", "Scores et seuils"),
    ("✅ Topologie réseau", "Graphe des connexions"),
    ("✅ Métriques de sécurité", "KPIs et statistiques"),
    ("✅ Palettes de couleurs", "Thèmes visuels")
]

for test, description in tests_results:
    print(f"  {test}: {description}")

print(f"\n📊 Données testées: {len(df):,} événements de sécurité")
print(f"🎯 Toutes les fonctionnalités du service de visualisation ont été testées avec succès!")
print(f"\n🔧 Le service est prêt pour l'intégration dans votre application Streamlit.")